In [18]:
from numpy import array, zeros, where, arange, linspace, exp, sin, cos, sort, sum, dot, ndarray, random
from numpy.random import uniform, choice, seed
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from mpl_toolkits import mplot3d

In [19]:
def train_test_split(dataframe, predictors, split=0.8, method_seed=None):
    from numpy.random import choice, seed
    if not method_seed == None:
        seed(method_seed)
    X = dataframe.drop(columns=predictors)
    Y = dataframe[predictors]
    split_count = int(len(X) * split)
    order = choice(len(X), len(X), replace=False)

    return (X.iloc[order[:split_count]], Y.iloc[order[:split_count]], X.iloc[order[split_count:]], Y.iloc[order[split_count:]])


## Lets Build a Neural Network :)

Lets describe the equations! $\beta$ and $w$ are weights (bias and scalar respectively), $X = \{x_1, x_2, \ldots, x_p\}$ are the data for $p$ neurons in the previous layer, $A_{l,k}$ is the $k$th neuron in the $l$th hidden layer, $A_{l - 1, j}$ is the value of the jth neuron of the previous layer, $g(z)$ is the activation function, $f(x)$ is the output, and $N$ is the number of data points.
$$A_{l,k}(x) = g(\beta_{l,k} + \sum_{j=1}^pw_{l,k,j}A_{l - 1, j})$$

$f(x)$ is the same but with an extra bias term and sometimes without an activation function:
$$f_k(x) = \alpha_{final,k} + g(\beta_{l,k} + \sum_{j=1}^pw_{l,k,j}A_j)$$
$$f_k(x) = \beta_{l,k} + \sum_{j=1}^pw_{l,k,j}A_j$$

These functions describe the model but not how to *train* the model. Do do so, we need a loss function and backpropagation. We will be using L2 loss:
$$R(\theta) = \frac{1}{N}\sum^N_{i=1}(y - f_\theta(X_i))^2$$
$$\frac{\partial R}{\partial \theta_i} =  -2\sum^N_{i=1}(y - f_\theta(X_i))*\frac{\partial f_\theta(X_i)}{\partial \theta_i}$$
$$\frac{\partial f_\theta(X_i)}{\partial \theta_i} = \frac{\partial}{\partial \theta_i} (\alpha_{final,k} + g(\beta_{l,k} + \sum_{j=1}^pw_{l,k,j}A_j))$$



In [41]:
class NeuralNetwork:
    '''
    This Class handles the construction, tuning, and use of a neural network.

    Intended Usage:
    model = NeuralNetwork()

    # Set the number of input neurons
    model.set_input_layer(num_neurons = 2)

    # Add hidden layer
    model.add_hidden_layer(num_neurons = 4)

    # set output layer
    model.set_output_layer(num_neurons = 1)

    # Initial prediction w/ random weights
    print(f'Model predicts: {model.forward_pass(test_data)}')

    # Train the model
    epochs = 1000
    for epoch in range(epochs):
        model.backpropagation(train_data)

    # Next prediction
    print(f'Model predicts: {model.forward_pass(test_data)}')
    '''

    def __relu(x):
      if isinstance(x, (list, ndarray)):
          return [i if i>0 else 0 for i in x]
      return x if x>0 else 0

    def __relu_deriv(x):
      if isinstance(x, (list, ndarray)):
          return [1 if i>0 else 0 for i in x]
      return 1 if x>0 else 0

    # Constructor
    def __init__(self, input_neurons, def_weight_range = (-2,2)):
        # Class Properties
        self.bias_weights = list()
        self.scalar_weights = list()
        self.neuron_values = [zeros(input_neurons) - 1]
        self.deriv_values = list()
        self.act_funcs = list()
        self.final_biases = None
        self.def_weight_range = def_weight_range

    # Add Layers
    def add_layer(self, neurons, weights = None, act_func= __relu, act_func_deriv= __relu_deriv):
        num_last_layer = len(self.neuron_values[-1])
        if weights == None:
            self.bias_weights += [uniform(self.def_weight_range[0], self.def_weight_range[1], neurons)]
            self.scalar_weights += [uniform(self.def_weight_range[0], self.def_weight_range[1], (neurons, num_last_layer))]
            self.neuron_values += [zeros(neurons) - 1]
            self.deriv_values += [zeros(neurons)]
            self.act_funcs += [[act_func, act_func_deriv]]
        else:
            raise Exception("Strict assignment of weights not yet implemented")
        
    # set output layer
    def set_output_layer(self, neurons, weights = None, act_func = __relu, act_func_deriv = __relu_deriv):
        num_last_layer = len(self.neuron_values[-1])
        if weights == None:
            self.bias_weights += [uniform(self.def_weight_range[0], self.def_weight_range[1], neurons)]
            self.scalar_weights += [uniform(self.def_weight_range[0], self.def_weight_range[1], (neurons, num_last_layer))]
            self.neuron_values += [zeros(neurons) - 1]
            self.deriv_values += [zeros(neurons)]
            self.act_funcs += [[act_func, act_func_deriv]]
            self.final_biases = uniform(self.def_weight_range[0], self.def_weight_range[1], neurons)
        else: 
            raise Exception("Strict assignment of weights in layer initialization not yet implemented")

    def predict(self, X):
        neuron_values = list()
        neuron_values += [X]

        for layer in range(1, len(self.neuron_values)):
            act_func = self.act_funcs[layer-1][0]
            unmodded_values = array([
                dot(neuron_values[layer - 1], self.scalar_weights[layer - 1][neuron]) 
                for neuron in 
                range(len(self.neuron_values[layer]))
                ])
            As = act_func(self.bias_weights[layer - 1] + unmodded_values)
            neuron_values += [As]
        
        return neuron_values[-1] + self.final_biases

    def forward_pass(self, X):
        
        # Handle input dropout
        X = X if isinstance(X, ndarray) else array(X)
        dropped_neurons = self.dropped_neurons[0]
        self.neuron_values[0] = X
        dropout_adjustment = 1
        if len(dropped_neurons) > 0:
            self.neuron_values[0][dropped_neurons] = 0.0
            dropout_adjustment = len(X) / len(dropped_neurons)

        # Handle Hidden Layers
        for layer in range(1, len(self.neuron_values) - 1):
            act_func = self.act_funcs[layer - 1][0]
            act_func_deriv = self.act_funcs[layer - 1][1]
            unmodded_values = array([
                dot(self.neuron_values[layer - 1], self.scalar_weights[layer - 1][neuron] * dropout_adjustment) 
                for neuron in 
                range(len(self.neuron_values[layer]))
            ])

            self.neuron_values[layer] = act_func(self.bias_weights[layer - 1] * dropout_adjustment + unmodded_values)
            self.deriv_values[layer - 1] = act_func_deriv(self.bias_weights[layer - 1] * dropout_adjustment + unmodded_values)

            # Handle Hidden layer Dropout
            dropped_neurons = self.dropped_neurons[layer]
            dropout_adjustment = 1
            if len(dropped_neurons) > 0:
                dropout_adjustment = len(self.neuron_values[layer]) / len(dropped_neurons)
                print(f'dropped neurons array type: {dropped_neurons.dtype}')
                self.neuron_values[layer][dropped_neurons] = 0.0
                self.deriv_values[layer - 1][dropped_neurons] = 0.0

        # Handle Final layer
        act_func = self.act_funcs[-1][0]
        act_func_deriv = self.act_funcs[-1][1]
        unmodded_values = array([
            dot(self.neuron_values[-2], self.scalar_weights[-1][neuron]) 
            for neuron in 
            range(len(self.neuron_values[-1]))
        ])

        self.neuron_values[-1] = self.final_biases + act_func(self.bias_weights[-1] + unmodded_values)
        self.deriv_values[-1] = act_func_deriv(self.bias_weights[-1] + unmodded_values)

        return self.neuron_values[-1]

    def __str__(self):
        print(f'Bias Weights: \n      {self.bias_weights}')
        print(f'Scalar Weights: \n      {self.scalar_weights}')
        print(f'Final Weights: \n      {self.final_biases}')
        print(f'Neuron Values: \n      {self.neuron_values}')
        print(f'Deriv Values: \n      {self.deriv_values}')
        return ""

    def set_training_data(self, x, y):
        if x.shape[1] != len(self.neuron_values[0]):
            raise Exception("Data and input layer have different dimensions!")
        y_output_size = 1 if len(y.shape) == 1 or y.shape[1] == 1 else y.shape[1]
        if y_output_size != len(self.neuron_values[-1]):
            raise Exception("Data and output layer have different dimensions!")
        if x.shape[0] != y.shape[0]:
            raise Exception("X and Y must be equal!")
        self.x_training = x
        self.y_training = y
    
    def backpropagation(self, learning_rate=0.001, batch=20, seed=42):
        num_layers = len(self.neuron_values)
        neurons_per_layer = [len(x) for x in self.neuron_values]
        LR = learning_rate / batch

        random.seed(seed)

        x_batch = self.x_training[choice(len(self.x_training), batch)]
        y_batch = self.y_training[choice(len(self.y_training), batch)]

        # Weight Buffers
        bias_weight_b = [zeros(neurons_per_layer[layer]) for layer in range(1, num_layers)]
        scalar_weight_b = [zeros((neurons_per_layer[layer], neurons_per_layer[layer-1])) for layer in range(1, num_layers)]
        final_weight_b = zeros(neurons_per_layer[-1])


        for X,Y in zip(x_batch, y_batch):

            # Calculate Neuron values and derivatives
            y_pred = self.forward_pass(X)

            # Get the difference between predicted and observed values
            R = (y_pred - Y) / batch

            # Final Biases
            final_weight_b += -1 * LR * R

            # Final Layer inner Biases
            bias_weight_b[-1] += -1 * LR * R * self.deriv_values[-1]

            # Scalar Weights
            if neurons_per_layer[-1] == 1:
                scalar_weight_b[-1] += -1 * LR * R * self.deriv_values[-1] * self.neuron_values[-2]
            else:
                # TODO following list comprehension is suspect
                scalar_weight_b[-1] += -1 * LR * R * array([self.deriv_values[-1][j] * self.neuron_values[-2] for j in range(neurons_per_layer[-1])])

            # Currently partial f / partial f hence an array of ones...
            pfpLast = zeros(neurons_per_layer[-1]) + 1

            # optimize hidden layer weights
            for layer in range(num_layers - 2, 0, -1):

                # partial DS layer / partial current layer Calculation - calculated for ea neuron in current layer
                pLastpAs = [self.deriv_values[layer] * self.scalar_weights[layer][:,neuron] for neuron in range(neurons_per_layer[layer])]

                # partial f(x) / partial current layer - calculated using dot(partial f / partial last, partial last / partial A)
                pfpAs = [dot(pfpLast, pLastpA) for pLastpA in pLastpAs]

                # update partial f(x) / partial US layer to match partial f(x) / partial current layer for next loop
                pfpLast = pfpAs

                # bias weights
                bias_weight_b[layer - 1] += -1 * LR * R * pfpAs * self.deriv_values[layer - 1]

                # scalar weights
                pfpOmegas = array([pfpAs[k] * self.deriv_values[layer-1][k] * self.neuron_values[layer-1] for k in range(neurons_per_layer[layer])])
                scalar_weight_b[layer - 1] += -1 * LR * R * pfpOmegas

        self.final_biases += final_weight_b
        for layer in range(num_layers-1):
            self.bias_weights[layer] += bias_weight_b[layer]
            self.scalar_weights[layer] += scalar_weight_b[layer]

    def set_dropout_rates(self, rates, seed=None):
        self.dropout_rates = rates
        self.cycle_dropped_neurons(seed=None)
    
    def cycle_dropped_neurons(self, seed=None):
        if seed != None:
            random.seed(seed)
        self.dropped_neurons = list()
        for layer in range(len(self.neuron_values) - 1):
            layer_len = len(self.neuron_values[layer])
            self.dropped_neurons += [choice(layer_len, int(self.dropout_rates[layer] * layer_len))]
        
    def set_scalar_weights(self, layer, weights):
        weights = weights if isinstance(weights, ndarray) else array(weights)
        self.scalar_weights[layer - 1] = weights.astype(float)
    
    def set_bias_weights(self, layer, weights):
        weights = weights if isinstance(weights, ndarray) else array(weights)
        self.bias_weights[layer - 1] = weights.astype(float)
    
    def set_final_biases(self, weights):
        weights = weights if isinstance(weights, ndarray) else array(weights)
        self.final_biases = weights.astype(float)

    def save_weights(self, filepath):
        
        layers = len(self.neuron_values)

        # Final Weights
        file = "-Final Weights-\n"
        file += f'  [{",".join([x for x in self.final_biases.astype(str)])}]\n'

        # Bias Weights
        file += "-Bias Weights-\n"
        for layer in range(1, layers):
            file += f'  [{",".join([x for x in self.bias_weights[layer-1].astype(str)])}]\n'

        # scalar weights
        file += "-Scalar Weights-\n"
        for layer in range(1, layers):
            file += '  [\n'
            temp = ""
            for neuron in range(len(self.neuron_values[layer])):
                temp += f'    [{",".join([x for x in self.scalar_weights[layer-1][neuron].astype(str)])}],'
                temp += "" if neuron == len(self.neuron_values[layer]) - 1 else "\n"
            file += f'{temp}\n  ],\n'

        with open(filepath, 'w') as f:
            f.write(file)

    def load_weights(self, filepath):
        from re import compile
        re_values = compile('(-?\\d+\\.?\\d*),?')
        re_new_layer = compile('\\s+],')

        with open(filepath, 'r') as f:

            f.readline()
            line = f.readline()
            while "bias" not in line.lower():
                self.final_biases = array(re_values.findall(line), dtype=float)
                line = f.readline()
            
            line = f.readline()
            layer = 0
            while "scalar" not in line.lower():
                self.bias_weights[layer] = array(re_values.findall(line), dtype=float)
                layer += 1
                line = f.readline()
            
            f.readline()
            layer = 0
            neuron = 0
            line = f.readline()
            while line != "":
                if re_new_layer.match(line) != None:
                    neuron = 0
                    layer += 1
                    f.readline()
                else:
                    self.scalar_weights[layer][neuron] = array(re_values.findall(line), dtype=float)
                    neuron += 1
                line = f.readline()

    def rmse(self, batch = 10, seed=42):
        random.seed(seed)

        x_batch = self.x_training[choice(len(self.x_training), batch)]
        y_batch = self.y_training[choice(len(self.y_training), batch)]

        return (sum([(Y - self.predict(X))**2 for X,Y in zip(x_batch, y_batch)], axis=0) / batch)**(1/2)


## Test Neural Network Framework

In [34]:
# Activation Functions

sigmoid = lambda x: 1 / (1 + exp(-x))
sigmoid_deriv = lambda x: sigmoid(x)*(1 - sigmoid(x))

def ca1(x):
    return sin(x) / x
def ca1_deriv(x):
    return (cos(x) * x - sin(x)) / x**2

identity = lambda x: x
identity_deriv = lambda x: 1

def relu(x):
    if type(x) in (ndarray, list):
        return array([0 if i < 0 else i for i in x])
    return 0 if x < 0 else x

def relu_deriv(x):
    if type(x) in (ndarray, list):
        return array([0 if i < 0 else 1 for i in x])
    return 0 if x < 0 else 1

In [35]:
random.seed(42)
x = uniform(-10, 10, 5000)
y = uniform(-10, 10, 5000)

f = lambda x,y: x**2 + y**2

z = [f(x,y) for x,y in zip(x,y)]

In [23]:
new_NN = NeuralNetwork(3, def_weight_range=(-5, 5))
new_NN.add_layer(12, act_func=relu, act_func_deriv=relu_deriv)
new_NN.add_layer(6, act_func=ca1, act_func_deriv=ca1_deriv)
new_NN.set_output_layer(1, act_func=identity, act_func_deriv=identity_deriv)


new_var = [(x**2 + y**2)**(1/2) for x,y in zip(x[:500],y[:500])]

new_NN.set_training_data(x=array([x[:500],y[:500],new_var]).T, y=array(z)[:500])

In [24]:
new_NN.set_dropout_rates([0,0,0])
init_rmse = new_NN.rmse(batch=200)
print(f'initial RMSE: {init_rmse}')
LR = .02
lr_floor = 0.0001
epochseed = uniform(0, 10000, 5).astype(int)
run_count = 0

for seed in epochseed:
    if run_count % 100 == 0:
        print(f'{(run_count / len(epochseed)) * 100:.2f}% done')
    if run_count == 40:
        new_NN.set_dropout_rates([1/3, 1/6, 1/6])
        new_NN.cycle_dropped_neurons()
    if run_count > 40 and run_count % 6 == 0:
        new_NN.cycle_dropped_neurons()
    new_NN.backpropagation(learning_rate=LR, batch=64, seed=seed)
    LR = LR * 0.95 if LR > lr_floor else lr_floor
    run_count += 1
new_rmse = new_NN.rmse(batch=200)
print(f'RMSE diff: {new_rmse - init_rmse}, new RMSE: {new_rmse}')

initial RMSE: [79.39349003]
0.00% done
RMSE diff: [-0.15124738], new RMSE: [79.24224265]


## Hitters Data  
This example comes from the book **An Introduction to Statistical Learning** by James, Gareth et. al. (2023). The example starts on page 437. I'm very confused. It doesn't specify what we're trying to predict—it doesn't even give us a summary of what the Hitters dataset is! More work could have gone into writing the labs in this book.

I'm going to go ahead and do salary prediction.

In [25]:
hitters_data = pd.read_csv('Hitters.csv')
hitters_data = pd.get_dummies(
    hitters_data, 
    columns=['League', 'Division', 'NewLeague']).dropna()
hitters_data.head()

,AtBat,Hits,HmRun,Runs,RBI,Walks,Years,CAtBat,CHits,CHmRun,...,PutOuts,Assists,Errors,Salary,League_A,League_N,Division_E,Division_W,NewLeague_A,NewLeague_N
1,315,81,7,24,38,39,14,3449,835,69,...,632,43,10,475.0,False,True,False,True,False,True
2,479,130,18,66,72,76,3,1624,457,63,...,880,82,14,480.0,True,False,False,True,True,False
3,496,141,20,65,78,37,11,5628,1575,225,...,200,11,3,500.0,False,True,True,False,False,True
4,321,87,10,39,42,30,2,396,101,12,...,805,40,4,91.5,False,True,True,False,False,True
5,594,169,4,74,51,35,11,4408,1133,19,...,282,421,25,750.0,True,False,False,True,True,False


## Lets Set up Train and Test Sets
We'll do an 80% train, 20% test split.

In [36]:
(X_Train, Y_Train, X_Test, Y_Test) = train_test_split(hitters_data, 'Salary', split=0.8, method_seed=42)
X_Train = X_Train.to_numpy()
Y_Train = Y_Train.to_numpy()
X_Test = X_Test.to_numpy()
Y_Test = Y_Test.to_numpy()


In [42]:
Hitter_NN = NeuralNetwork(22, def_weight_range=(-5,5))
Hitter_NN.add_layer(48)
Hitter_NN.add_layer(12)
Hitter_NN.add_layer(5)
Hitter_NN.set_output_layer(1)

Hitter_NN.set_training_data(x=X_Train, y=Y_Train)

Hitter_NN.set_dropout_rates([8/22, 1/4, 3/12, 1/5])

In [43]:
Hitter_NN.predict(X_Test[0])

array([4.36317947])

In [45]:
init_rmse = Hitter_NN.rmse(50)
print(f'Initial RMSE: {init_rmse}')
epochs = 100
epochseed = uniform(0, 10000, epochs).astype(int)
LR = 3
LR_floor = 0.001
for epoch in epochseed:
    Hitter_NN.backpropagation(LR, batch=20, seed=epoch)
    LR = LR * 0.90 if LR > LR_floor else LR_floor
print(f'New RMSE: {Hitter_NN.rmse(50)}')


Initial RMSE: [330053.94608258]
dropped neurons array type: int64


TypeError: only integer scalar arrays can be converted to a scalar index